In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
import itertools

In [2]:
load_dotenv()
user = os.getenv("MYSQL_USER")
password = quote_plus(os.getenv("MYSQL_PASSWORD"))
host = os.getenv("MYSQL_HOST")
db = os.getenv("MYSQL_DB")
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{db}")

df = pd.read_sql("SELECT * FROM daily_demand_features", con=engine)
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
print(df.shape)

(371053, 10)


In [3]:
def build_features(df, freq, top_n):
    top_products = df.groupby("stockcode")["daily_qty"].sum().sort_values(ascending=False).head(top_n).index
    sub = df[df["stockcode"].isin(top_products)].copy()

    agg = (
        sub.groupby(["stockcode", "description", pd.Grouper(key="invoicedate", freq=freq)])
        ["daily_qty"].sum()
        .reset_index()
        .rename(columns={"daily_qty": "qty", "invoicedate": "period"})
    )

    agg = agg.sort_values(["stockcode", "period"])
    agg["month"] = agg["period"].dt.month
    agg["lag_1"] = agg.groupby("stockcode")["qty"].shift(1)
    agg["lag_2"] = agg.groupby("stockcode")["qty"].shift(2)
    agg["lag_3"] = agg.groupby("stockcode")["qty"].shift(3)
    agg["rolling_mean_3"] = agg.groupby("stockcode")["qty"].transform(lambda x: x.rolling(3, min_periods=1).mean())
    agg["rolling_std_3"] = agg.groupby("stockcode")["qty"].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0)
    agg["ewm_mean"] = agg.groupby("stockcode")["qty"].transform(lambda x: x.ewm(span=3).mean())

    agg = agg.dropna(subset=["lag_1", "lag_2", "lag_3"])
    agg["log_qty"] = np.log1p(agg["qty"])
    return agg

In [11]:
def train_and_evaluate(agg, n_test_periods):
    cutoff = sorted(agg["period"].unique())[-n_test_periods]
    train = agg[agg["period"] < cutoff]
    test = agg[agg["period"] >= cutoff]

    features = ["month", "lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3", "ewm_mean"]
    X_train, y_train = train[features], train["log_qty"]
    X_test, y_test = test[features], test["log_qty"]

    param_grid = {"n_estimators": [200, 400], "max_depth": [3, 4, 5], "learning_rate": [0.03, 0.05, 0.1]}
    best_score, best_params = -np.inf, None
    tscv = TimeSeriesSplit(n_splits=3)

    for n_est, depth, lr in itertools.product(param_grid["n_estimators"], param_grid["max_depth"], param_grid["learning_rate"]):
        scores = []
        for tr_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            m = XGBRegressor(n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=42)
            m.fit(X_tr, y_tr)
            pred_val, actual_val = np.expm1(m.predict(X_val)), np.expm1(y_val)
            wape = np.sum(np.abs(pred_val - actual_val)) / np.sum(actual_val)
            scores.append(1 - wape)
        avg_score = np.mean(scores)
        if avg_score > best_score:
            best_score, best_params = avg_score, (n_est, depth, lr)

    n_est, depth, lr = best_params
    final_model = XGBRegressor(n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=42)
    final_model.fit(X_train, y_train)

    preds, actual = np.expm1(final_model.predict(X_test)), np.expm1(y_test)
    accuracy = 1 - (np.sum(np.abs(preds - actual)) / np.sum(actual))

    test = test.copy()
    test["predicted_qty"] = preds
    test["actual_qty"] = actual
    return accuracy, test, best_params, final_model

In [5]:
agg_weekly = build_features(df, "W", 100)
acc_weekly, test_weekly, params_weekly = train_and_evaluate(agg_weekly, n_test_periods=8)
print("Weekly accuracy:", acc_weekly, "| params:", params_weekly)

final_test, final_acc = None, acc_weekly

if acc_weekly >= 0.8:
    final_test = test_weekly
else:
    agg_monthly = build_features(df, "ME", 50)
    acc_monthly, test_monthly, params_monthly = train_and_evaluate(agg_monthly, n_test_periods=3)
    print("Monthly accuracy:", acc_monthly, "| params:", params_monthly)
    if acc_monthly >= 0.8:
        final_test, final_acc = test_monthly, acc_monthly
    else:
        final_acc = max(acc_weekly, acc_monthly)

Weekly accuracy: 0.8747548454293987 | params: (400, 5, 0.1)


In [6]:
if final_test is not None:
    forecast_output = final_test[[
        "stockcode", "description", "period", "actual_qty", "predicted_qty", "rolling_mean_3"
    ]].rename(columns={"period": "invoicedate", "rolling_mean_3": "rolling_mean_7"})

    forecast_output["reorder_flag"] = forecast_output.apply(
        lambda row: "Restock Needed" if row["predicted_qty"] > row["rolling_mean_7"] * 1.2 else "Sufficient Stock",
        axis=1
    )

    forecast_output.to_sql("demand_forecast", con=engine, if_exists="replace", index=False)
    print(f"Accuracy {final_acc:.2%} — pushed to MySQL")
else:
    print(f"Best accuracy reached: {final_acc:.2%} — below 80%, NOT pushed. See note below.")

Accuracy 87.48% — pushed to MySQL
